# Обобщённые линейные модели: регрессия для нетипичных откликов

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Обобщённые линейные модели».

## Что делает метод

Обобщённая линейная модель (GLM) переносит идею линейной регрессии на отклики, для которых обычная регрессия неприменима: счётные (число событий), долевые, бинарные. Модель состоит из трёх компонент: распределения отклика из экспоненциального семейства, линейного предиктора и функции связи, соединяющей среднее отклика с линейным предиктором.

Метод применим, когда отклик — это, например, число наблюдённых событий за интервал (пуассоновская регрессия) или доля. В этом блокноте мы построим пуассоновскую модель счётного отклика, сравним её с наивной линейной регрессией и разберём интерпретацию коэффициентов. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Обычная линейная регрессия предполагает, что отклик — непрерывное число с нормальным разбросом ошибок. Но в науке часто встречаются данные другой природы:
- **Счётные данные**: число видов на площадке, число мутаций в клетке, число аварий в месяц — всегда целые неотрицательные числа, причём их разброс растёт вместе со средним.
- **Долевые данные**: доля проросших семян, частота встречаемости вида — ограничены диапазоном [0, 1].
- **Бинарные данные**: событие произошло или нет — тот же случай, что у логистической регрессии.

Обобщённые линейные модели (GLM) расширяют идею линейной регрессии на все эти случаи. Вместо прямой связи «предиктор → отклик» они вводят **функцию связи** — математическую трубку-переходник, которая согласует линейный предиктор с подходящей шкалой отклика. Для счётных данных это логарифм: модель предсказывает log(среднее_число_событий) как линейную комбинацию предикторов. Это гарантирует, что предсказанное число событий всегда положительно.

**Ключевые термины:**
- **Функция связи** — преобразование, соединяющее линейный предиктор со средним отклика (например, log для счётных данных).
- **Пуассоновская регрессия** — GLM для счётных данных, предполагает, что отклик распределён по Пуассону: дисперсия равна среднему.
- **Коэффициент GLM** — интерпретируется на шкале функции связи; для log-связи его экспонента даёт мультипликативный эффект.
- **Сверхдисперсия** — когда реальная дисперсия отклика заметно превышает ожидаемую по пуассоновской модели; требует более сложной модели (отрицательная биномиальная).

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib statsmodels

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем счётные данные: отклик — число событий (например, число зарегистрированных проявлений признака у объекта), порождённое пуассоновским процессом, интенсивность которого экспоненциально зависит от двух предикторов. Это эталонная задача для пуассоновской GLM.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400
x1 = rng.normal(size=n)              # непрерывный предиктор
x2 = rng.uniform(-1, 1, size=n)      # непрерывный предиктор

# интенсивность процесса: log(mu) линейно зависит от предикторов
log_mu = 0.8 + 0.6 * x1 - 0.9 * x2
y = rng.poisson(np.exp(log_mu))      # счётный отклик

df = pd.DataFrame({'x1': x1, 'x2': x2, 'count': y})
print(f'Наблюдений: {n}')
print(f'Отклик: целые числа от {y.min()} до {y.max()}, '
      f'среднее {y.mean():.2f}, дисперсия {y.var():.2f}')
df.head()

### Наглядный «ага»-эксперимент: почему линейная регрессия неприменима к счётным данным

Прежде чем строить GLM, посмотрим на проблему наглядно. Гистограмма ниже показывает распределение нашего счётного отклика. Обратите внимание: правый хвост длинный, а значения — всегда целые и неотрицательные. Обычная линейная регрессия будет предсказывать непрерывные числа и может давать отрицательные значения — что бессмысленно для числа событий.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.2))

# Гистограмма отклика
axes[0].hist(y, bins=30, color=VIZ['series'][0],
             edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Число событий (счётный отклик)')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение счётного отклика')
axes[0].axvline(y.mean(), color=VIZ['series'][2],
                linestyle='--', label=f'среднее = {y.mean():.1f}')
axes[0].legend()

# Связь отклика с предиктором x1
sort_idx = np.argsort(x1)
from sklearn.linear_model import LinearRegression
lin_demo = LinearRegression().fit(x1.reshape(-1, 1), y)
x1_grid = np.linspace(x1.min(), x1.max(), 200)
y_lin_demo = lin_demo.predict(x1_grid.reshape(-1, 1))
# Экспоненциальная кривая (правда)
y_true_demo = np.exp(0.8 + 0.6 * x1_grid)

axes[1].scatter(x1, y, alpha=0.3, s=15,
                color=VIZ['series'][0], label='наблюдения')
axes[1].plot(x1_grid, y_true_demo, color=VIZ['series'][1],
             linewidth=2.5, label='истинная зависимость (экспоненциальная)')
axes[1].plot(x1_grid, y_lin_demo, color=VIZ['series'][2],
             linewidth=2.0, linestyle='--',
             label='линейная регрессия (некорректна)')

# Подсветка отрицательных прогнозов
neg_mask = y_lin_demo < 0
if neg_mask.any():
    axes[1].fill_between(x1_grid, y_lin_demo, 0,
                         where=neg_mask,
                         color=VIZ['series'][2], alpha=0.2,
                         label='отрицательные прогнозы (!)' )

axes[1].axhline(0, color=VIZ['edge'], linewidth=0.8, linestyle=':')
axes[1].set_xlabel('Предиктор x1')
axes[1].set_ylabel('Число событий')
axes[1].set_title('Счётный отклик: линейная vs. экспоненциальная связь')
axes[1].legend(loc='upper left', fontsize=9)

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый: гистограмма отклика. Характерные черты счётного распределения — правосторонний хвост (много нулей и малых значений, редко — большие числа) и скошенность. Среднее и дисперсия в пуассоновском распределении равны — проверим это в тексте вывода из ячейки данных.

Правый: синие точки — реальные наблюдения. Зелёная кривая — истинная экспоненциальная зависимость, которую нам нужно восстановить. Оранжевая пунктирная прямая — линейная регрессия: она уходит в отрицательную область (закрашена) и явно не соответствует природе данных. Именно здесь GLM с логарифмической функцией связи приходит на помощь.

## 4. Применение метода

Подберём пуассоновскую GLM с логарифмической функцией связи. Основной инструмент — `statsmodels`: он выдаёт таблицу коэффициентов со стандартными ошибками и p-значениями, привычную для научных публикаций. Если `statsmodels` недоступна, используется `PoissonRegressor` из `scikit-learn` как фолбэк.

**Как интерпретировать коэффициенты?** Для модели с log-связью: если коэффициент при x1 равен 0.6, то при увеличении x1 на 1 единицу ожидаемое число событий умножается на exp(0.6) ≈ 1.82, то есть растёт на 82%.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, random_state=42)

glm_summary = None
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    glm = smf.glm('count ~ x1 + x2', data=train,
                  family=sm.families.Poisson()).fit()
    glm_summary = glm.summary2().tables[1]
    print(glm_summary.round(3))
    predict_glm = lambda d: glm.predict(d)
except Exception as exc:
    print('statsmodels недоступна, фолбэк на scikit-learn:', exc)
    from sklearn.linear_model import PoissonRegressor
    pr = PoissonRegressor(alpha=0.0, max_iter=1000)
    pr.fit(train[['x1', 'x2']], train['count'])
    print('Свободный член:', round(pr.intercept_, 3))
    print('Коэффициенты x1, x2:', np.round(pr.coef_, 3))
    predict_glm = lambda d: pr.predict(d[['x1', 'x2']])

Вывод таблицы `statsmodels` содержит: столбец `Coef.` — оценки коэффициентов (на логарифмической шкале), `Std.Err.` — их неопределённость, `P>|z|` — значимость, `[0.025 / 0.975]` — доверительные интервалы. Истинные значения коэффициентов в нашем синтетическом примере: свободный член 0.8, x1 = 0.6, x2 = −0.9.

### Сравнение с наивной линейной регрессией

Обычная линейная регрессия игнорирует природу счётного отклика и может предсказывать отрицательные значения. Сравним обе модели по качеству прогноза на тестовой выборке.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

lin = LinearRegression().fit(train[['x1', 'x2']], train['count'])
pred_lin = lin.predict(test[['x1', 'x2']])
pred_glm = np.asarray(predict_glm(test))

print('Пуассоновская GLM:')
print(f'  средняя абс. ошибка: {mean_absolute_error(test["count"], pred_glm):.3f}')
print(f'  доля отрицательных прогнозов: {np.mean(pred_glm < 0):.0%}')
print('Линейная регрессия:')
print(f'  средняя абс. ошибка: {mean_absolute_error(test["count"], pred_lin):.3f}')
print(f'  доля отрицательных прогнозов: {np.mean(pred_lin < 0):.0%}')

### Визуализация прогноза

На графике слева — прогноз против факта для обеих моделей; справа — как предсказанная интенсивность зависит от первого предиктора при фиксированном втором.

**Как читать эти графики.**

Левый (прогноз против факта): точки вдоль пунктирной диагонали — хороший прогноз. Наблюдайте, что пуассоновская GLM (синие точки) держится вблизи диагонали, тогда как линейная регрессия (оранжевые точки) систематически отклоняется при больших значениях отклика.

Правый (предсказанная интенсивность): кривая показывает нелинейную (экспоненциальную) зависимость среднего числа событий от предиктора x1 при x2 = 0. Это прямое следствие логарифмической функции связи: линейная зависимость на логарифмической шкале становится экспоненциальной на исходной.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

axes[0].scatter(test['count'], pred_glm, alpha=0.6,
                color=VIZ['series'][0], label='пуассоновская GLM')
axes[0].scatter(test['count'], pred_lin, alpha=0.5,
                color=VIZ['series'][2], label='линейная регрессия')
lim = [0, test['count'].max() + 1]
axes[0].plot(lim, lim, color=VIZ['series'][3], linestyle='--')
axes[0].set_xlabel('Наблюдённое число событий')
axes[0].set_ylabel('Предсказанное значение')
axes[0].set_title('Прогноз против факта')
axes[0].legend()

grid = pd.DataFrame({'x1': np.linspace(-3, 3, 100), 'x2': 0.0})
axes[1].plot(grid['x1'], np.asarray(predict_glm(grid)),
             color=VIZ['series'][0])
axes[1].scatter(df['x1'], df['count'], alpha=0.25,
                color=VIZ['series'][3])
axes[1].set_xlabel('Предиктор x1')
axes[1].set_ylabel('Число событий')
axes[1].set_title('Предсказанная интенсивность (x2 = 0)')

fig.tight_layout()
plt.show()

## 5. Интерпретация результата

- **Коэффициенты GLM** интерпретируются на шкале функции связи. Для логарифмической связи экспонента коэффициента — мультипликативный эффект: при росте предиктора на единицу ожидаемое число событий умножается на это значение.
- **p-значения** (при наличии `statsmodels`) показывают значимость предикторов.
- **Сравнение с линейной регрессией**: пуассоновская модель не выдаёт отрицательных прогнозов и точнее описывает счётный отклик, поскольку учитывает рост дисперсии вместе со средним.
- **График интенсивности**: кривая показывает нелинейную (экспоненциальную) связь среднего отклика с предиктором, что и обеспечивает функция связи.

Если дисперсия отклика заметно превышает среднее (сверхдисперсия), вместо пуассоновской модели используйте отрицательную биномиальную.

## 6. Попробуйте на своих данных

Замените синтетический набор своими данными со счётным откликом (целые неотрицательные числа).

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже, укажите имя файла, столбец отклика и формулу модели.
3. Выполните ячейки разделов 4 и 5.

## 7. Поэкспериментируйте сами

1. **Проверьте сверхдисперсию.** В разделе 3 измените генерацию отклика: вместо `rng.poisson(np.exp(log_mu))` используйте `rng.negative_binomial(5, 5/(5+np.exp(log_mu)))`. Это отрицательное биномиальное распределение с той же средней, но большей дисперсией. Посмотрите, насколько ухудшится прогноз пуассоновской модели.

2. **Добавьте третий предиктор.** Создайте `x3 = rng.choice([0, 1], size=n)` (бинарная переменная, например, наличие обработки), добавьте в `log_mu` слагаемое `+ 0.4 * x3` и включите x3 в формулу модели. Значимо ли новый предиктор?

3. **Переведите коэффициенты в мультипликативные эффекты.** После подбора GLM выполните `import numpy as np; print(np.exp(glm.params))` — вы увидите, во сколько раз меняется ожидаемое число событий при изменении каждого предиктора на единицу.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# Отклик 'count' - целые неотрицательные числа.
# Формула модели в стиле statsmodels: 'count ~ x1 + x2 + x3'
#
# from sklearn.model_selection import train_test_split
# train, test = train_test_split(df, test_size=0.3, random_state=42)
# Далее повторите ячейки раздела 4, скорректировав формулу и
# список предикторов в фолбэке.

## 8. Краткие выводы

- GLM обобщают линейную регрессию на отклики, выходящие за пределы нормального распределения: счётные (Пуассон), бинарные (логистическая), долевые (бета-регрессия).
- Функция связи (log для Пуассона, logit для бинарного отклика) гарантирует, что предсказания остаются в разумном диапазоне (неотрицательны для счётных, от 0 до 1 для долей).
- Коэффициенты интерпретируются на шкале функции связи; для log-связи экспонента коэффициента — это мультипликативный эффект на среднее число событий.
- Если дисперсия отклика заметно превышает среднее (сверхдисперсия) — пуассоновская модель недооценивает неопределённость; используйте отрицательную биномиальную регрессию.
- GLM — прямое расширение знакомой линейной регрессии; освоив её логику, вы сможете применять GLM к широкому спектру научных данных.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Таблица `statsmodels` показала коэффициент при `x1` равным 0.58 (p < 0.001) и коэффициент при `x2` равным −0.87 (p < 0.001). Истинные значения — 0.6 и −0.9 соответственно. Как правильно интерпретировать коэффициент 0.58 на содержательном языке и что означает его знак?
2. Вы применили пуассоновскую GLM к реальным данным о числе наблюдений вида в точках маршрута. Дисперсия отклика оказалась в 4 раза больше среднего. Что это означает, какое предположение пуассоновской модели нарушено и каким методом следует заменить модель?
3. Линейная регрессия на тех же данных даёт MAE = 1.23, а пуассоновская GLM — MAE = 1.31. Коллега заключает, что линейная регрессия лучше. Оцените корректность этого вывода и назовите дополнительные критерии, по которым следует сравнивать модели для счётных данных.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Коэффициент 0.58 находится на логарифмической шкале (шкала функции связи). Содержательная интерпретация: при увеличении `x1` на одну единицу ожидаемое число событий умножается на exp(0.58) ≈ 1.79, то есть возрастает примерно на 79 %. Отрицательный знак у `x2` означает, что рост `x2` снижает ожидаемое число событий: exp(−0.87) ≈ 0.42 — каждая дополнительная единица `x2` сокращает ожидаемое число событий примерно вдвое.
2. Это сверхдисперсия (overdispersion): в пуассоновском распределении дисперсия равна среднему, а здесь она в 4 раза больше. Пуассоновская GLM недооценивает неопределённость, стандартные ошибки коэффициентов занижены, p-значения некорректны. Следует использовать отрицательную биномиальную регрессию (`statsmodels.formula.api.negativebinomial`), которая добавляет отдельный параметр дисперсии.
3. Вывод некорректен. MAE — метрика для непрерывного нормального отклика; для счётных данных она не отражает специфику задачи. Правильные критерии: (а) пуассоновское отклонение (deviance) или AIC — учитывают форму предполагаемого распределения; (б) доля отрицательных прогнозов у линейной модели — бессмысленных по природе задачи; (в) калиброванность: соответствие предсказанных частот наблюдённым по децилям; (г) содержательная обоснованность модели, согласованная с природой данных.
</details>
